# 02 — Retrieval Evaluation

**Workstream:** RAG · **Batch 6 (Day 6)** · **Status: IMPLEMENTED**

Fixed evaluation query set covering eligibility / benefits / off-topic / abstention categories
(Knowledge 12 §6), scored against the real 3-scheme index for top-1 correctness.

In [1]:
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai" / "common" / "config.py").exists())
sys.path.insert(0, str(_root))
from ai.rag.src.vector_store import FaissStore
from ai.rag.src.retriever import retrieve

store = FaissStore.load(_root / "ai/rag/models/scheme_index_v1")
print(f"Loaded index: {store.index.ntotal} vectors")

Loaded index: 15 vectors


## Eval set (query, expected top scheme)

In [2]:
EVAL_SET = [
    ("How much does PM-KISAN pay per year?", "pm-kisan"),
    ("Is there a fee to apply for KCC?", "kcc"),
    ("What is the premium for Kharif crop insurance?", "pmfby"),
    ("Can government employees get PM-KISAN?", "pm-kisan"),
    ("What is the interest rate on a Kisan Credit Card?", "kcc"),
    ("How do I report crop damage from hail?", "pmfby"),
]

correct = 0
for query, expected in EVAL_SET:
    results = retrieve(store, query, k=1)
    got = results[0]["scheme_id"] if results else None
    ok = got == expected
    correct += ok
    print(f"{'OK ' if ok else 'MISS'}  '{query}'  -> got={got}  expected={expected}  score={results[0]['score']:.3f}")

print(f"\nTop-1 accuracy: {correct}/{len(EVAL_SET)} = {correct/len(EVAL_SET):.1%}")

MISS  'How much does PM-KISAN pay per year?'  -> got=ap-annadata-sukhibhava  expected=pm-kisan  score=0.179
OK   'Is there a fee to apply for KCC?'  -> got=kcc  expected=kcc  score=0.087
OK   'What is the premium for Kharif crop insurance?'  -> got=pmfby  expected=pmfby  score=0.224
MISS  'Can government employees get PM-KISAN?'  -> got=ap-annadata-sukhibhava  expected=pm-kisan  score=0.135
OK   'What is the interest rate on a Kisan Credit Card?'  -> got=kcc  expected=kcc  score=0.234
OK   'How do I report crop damage from hail?'  -> got=pmfby  expected=pmfby  score=0.180

Top-1 accuracy: 4/6 = 66.7%


## Off-topic query — should score low (abstention candidate, tested properly in notebook 03)

In [3]:
off_topic = retrieve(store, "What is the weather forecast for tomorrow in Vijayawada?", k=1)
print(f"Off-topic query top score: {off_topic[0]['score']:.3f}  (compare to on-topic scores above - should be lower)")

Off-topic query top score: 0.000  (compare to on-topic scores above - should be lower)


## Next: `03_generation_grounding_abstention.ipynb`

**Status: IMPLEMENTED.** Real retrieval accuracy measured above on the real (if small, 3-scheme) index -
this number will change as the corpus grows in later batches, and is re-measured each time, not assumed.